In [1]:
# 2. Scaricare i dataset nella VM
import gdown
import os

folder_id = "1kDMoujUUYc0DbAMw4WGRtlXcruEVFCEd"
output_dir = "/content/datasets"
gdown.download_folder(id=folder_id, output=output_dir, quiet=False, use_cookies=False)
hdf5_files = [f for f in os.listdir(output_dir) if f.endswith('.hdf5')]
print("Dataset trovati:", hdf5_files)

Retrieving folder contents


Processing file 1A6ZR2owiAD4g9mp9t44BNYp3WKwsNGCi agnews-mxbai-public.hdf5
Processing file 1hMtlrggBTz13GiHzr2sUVyKVw8xEJn7b celeba-resnet-public.hdf5
Processing file 16XXe-L3eGg-qXwBhcYZ8jYIq8g78Y0k_ gooaq-distilroberta-public.hdf5
Processing file 17Ar1twEEp4YtNTAp355VHAT4nUeU8CbR imagenet-clip-public.hdf5
Processing file 1a4XKr-MkMEpCdS7dHFWkA-BGBGTDzqMC landmark-nomic-public.hdf5
Processing file 1WiJmokTtWiDHvHuISViCBBo0Atjv_Swd simplewiki-openai-public.hdf5
Processing file 1-NcAwWXjClOuwswh0MXoOgCYjDad4voB yahoo-minilm-public.hdf5


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1A6ZR2owiAD4g9mp9t44BNYp3WKwsNGCi
From (redirected): https://drive.google.com/uc?id=1A6ZR2owiAD4g9mp9t44BNYp3WKwsNGCi&confirm=t&uuid=4995c792-8e25-49fd-a9b1-4f42b61501b7
To: /content/datasets/agnews-mxbai-public.hdf5
100%|██████████| 3.15G/3.15G [01:16<00:00, 41.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1hMtlrggBTz13GiHzr2sUVyKVw8xEJn7b
From (redirected): https://drive.google.com/uc?id=1hMtlrggBTz13GiHzr2sUVyKVw8xEJn7b&confirm=t&uuid=b9fe8fe7-c956-40fd-a0f9-66870e59edf0
To: /content/datasets/celeba-resnet-public.hdf5
100%|██████████| 1.65G/1.65G [00:35<00:00, 46.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=16XXe-L3eGg-qXwBhcYZ8jYIq8g78Y0k_
From (redirected): https://drive.google.com/uc?id=16XXe-L3eGg-qXwBhcYZ8jYIq8g78Y0k_&confirm=t&uuid=ccfc8283-3ab4-4993-8218-60dd1ad0

Dataset trovati: ['gooaq-distilroberta-public.hdf5', 'landmark-nomic-public.hdf5', 'yahoo-minilm-public.hdf5', 'celeba-resnet-public.hdf5', 'agnews-mxbai-public.hdf5', 'simplewiki-openai-public.hdf5', 'imagenet-clip-public.hdf5']



Download completed


In [2]:
# 3. Creare directory di lavoro e file C++
!mkdir -p /content/ann_project
%cd /content/ann_project

/content/ann_project


In [3]:
!pip install pybind11 gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 6.3 MB/s eta 0:00:00


In [4]:
%%writefile linear_index.cpp
#include <pybind11/pybind11.h>
#include <pybind11/numpy.h>
#include <pybind11/stl.h>
#include <vector>
#include <cstdint>
#include <algorithm>
#include <cmath>

namespace py = pybind11;

class LinearIndex {
private:
    std::vector<float> data_;
    size_t npts_ = 0;
    size_t dim_ = 0;
    size_t total_distances_ = 0;

public:
    LinearIndex() = default;

    void fit(py::array_t<float> arr) {
        py::buffer_info buf = arr.request();
        if (buf.ndim != 2)
            throw std::runtime_error("Input deve essere un array 2D");
        npts_ = buf.shape[0];
        dim_ = buf.shape[1];
        float* ptr = static_cast<float*>(buf.ptr);
        data_.assign(ptr, ptr + npts_ * dim_);
    }

    py::array_t<int64_t> query(py::array_t<float> q, int k) {
        if (dim_ == 0 || npts_ == 0)
            throw std::runtime_error("Indice non addestrato");

        py::buffer_info qbuf = q.request();
        if (qbuf.ndim != 1 || qbuf.shape[0] != dim_)
            throw std::runtime_error("Query deve essere un vettore 1D di dimensione corretta");

        float* q_ptr = static_cast<float*>(qbuf.ptr);

        std::vector<std::pair<float, int>> dist_id(npts_);
        for (size_t i = 0; i < npts_; ++i) {
            float sum = 0.0f;
            for (size_t d = 0; d < dim_; ++d) {
                float diff = data_[i * dim_ + d] - q_ptr[d];
                sum += diff * diff;
            }
            dist_id[i] = {sum, static_cast<int>(i)};
        }

        total_distances_ += npts_;

        if (k > static_cast<int>(npts_)) k = npts_;
        std::nth_element(dist_id.begin(), dist_id.begin() + (k - 1), dist_id.end(),
                         [](const auto& a, const auto& b) { return a.first < b.first; });

        std::vector<int64_t> result(k);
        for (int i = 0; i < k; ++i)
            result[i] = dist_id[i].second;

        return py::array_t<int64_t>({k}, result.data());
    }

    size_t total_distances_count() const { return total_distances_; }
};

PYBIND11_MODULE(linear_ann_cpp, m) {
    py::class_<LinearIndex>(m, "LinearIndex")
        .def(py::init<>())
        .def("fit", &LinearIndex::fit)
        .def("query", &LinearIndex::query)
        .def("total_distances_count", &LinearIndex::total_distances_count);
}

Writing linear_index.cpp


In [6]:
%%writefile setup.py
from setuptools import setup
from pybind11.setup_helpers import Pybind11Extension, build_ext

ext_modules = [
    Pybind11Extension(
        "linear_ann_cpp",
        ["linear_index.cpp"],
        cxx_std=14,
    ),
]

setup(
    name="linear_ann_cpp",
    version="0.1",
    author="Tu",
    description="Modulo ANN lineare",
    ext_modules=ext_modules,
    cmdclass={"build_ext": build_ext},
)

Writing setup.py


In [7]:
!python setup.py build_ext --inplace

running build_ext
building 'linear_ann_cpp' extension
creating build/temp.linux-x86_64-cpython-312
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -I/usr/local/lib/python3.12/dist-packages/pybind11/include -I/usr/include/python3.12 -c linear_index.cpp -o build/temp.linux-x86_64-cpython-312/linear_index.o -fvisibility=hidden -g0 -std=c++14
linear_index.cpp: In member function ‘pybind11::array_t<long int> LinearIndex::query(pybind11::array_t<float>, int)’:
linear_index.cpp:36:45: warning: comparison of integer expressions of different signedness: ‘__gnu_cxx::__alloc_traits<std::allocator<long int>, long int>::value_type’ {aka ‘long int’} and ‘size_t’ {aka ‘long unsigned int’} []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wsign-compare-Wsign-compare]8;;]
   36 |         if (qbuf.ndim != 1 || qbuf.shape[0] != dim_)
creating build/lib.linux-x86_64-cpython

In [8]:
!ls -la *.so

-rwxr-xr-x 1 root root 257472 Mar  4 14:55 linear_ann_cpp.cpython-312-x86_64-linux-gnu.so


In [9]:
# Assicurati che il path al modulo sia nel sys.path
import sys
sys.path.insert(0, '/content/ann_project')

# Importa il modulo C++ compilato
import linear_ann_cpp

import numpy as np
import h5py
import os

class LinearANN:
    def __init__(self):
        self.index = linear_ann_cpp.LinearIndex()
        self._data = None

    def fit(self, data):
        self._data = data
        self.index.fit(data)

    def query(self, q, k):
        q_np = np.asarray(q, dtype=np.float32)
        return self.index.query(q_np, k)

    def total_distances_count(self):
        return self.index.total_distances_count()

# Test su un dataset
output_dir = "/content/datasets"
hdf5_files = [f for f in os.listdir(output_dir) if f.endswith('.hdf5')]
if hdf5_files:
    first = os.path.join(output_dir, hdf5_files[0])
    print(f"Elaborazione: {first}")
    with h5py.File(first, 'r') as f:
        train = f['/train'][:]
        queries = f['/test'][:5]   # prime 5 query per test rapido

    ann = LinearANN()
    ann.fit(train)

    for i, q in enumerate(queries):
        ids = ann.query(q, 100)
        print(f"Query {i}: primi 5 ID = {ids[:5]}")

    print("Distanze totali calcolate:", ann.total_distances_count())
else:
    print("Nessun dataset trovato in /content/datasets. Prima esegui lo scaricamento con gdown.")

Elaborazione: /content/datasets/gooaq-distilroberta-public.hdf5
Query 0: primi 5 ID = [1441361   67259  668629  864071  993248]
Query 1: primi 5 ID = [1471945  399068  544749 1221580  193095]
Query 2: primi 5 ID = [ 994492  721277 1069051  749837 1444732]
Query 3: primi 5 ID = [1018298  851793  829650 1129629  804321]
Query 4: primi 5 ID = [1306861  340806 1106290   18552  914409]
Distanze totali calcolate: 7370120
